# Lab 5 — Completed Exercises

**Student:** Abdulaziz Saqer BinRuqush  
**Student ID:** 2230007229

Instructor notebook: [Lab5_Text_Representation-2.ipynb](https://github.com/MohammedMosuily/NLP_Lab5)  
Source commit: `0b87407545bbf8d6d1a81c97690af16593d7be27`

Run the setup commands in the course README before running all cells.
Instructor examples and task instructions are retained below.


In [1]:
from pathlib import Path
import os
import sys
import random
import numpy as np
import pandas as pd
from IPython.display import display

LAB_DIR = Path.cwd()
if not (LAB_DIR / 'Lab5_Text_Representation-2.ipynb').is_file():
    LAB_DIR = LAB_DIR / "Lab5"
if not (LAB_DIR / 'Lab5_Text_Representation-2.ipynb').is_file():
    raise FileNotFoundError("Run from the course root or this lab's folder.")
PROJECT_ROOT = LAB_DIR.parent
os.environ.setdefault("KAGGLEHUB_CACHE", str(PROJECT_ROOT / ".cache" / "kagglehub"))
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache" / "huggingface"))
os.environ.setdefault("NLTK_DATA", str(PROJECT_ROOT / ".cache" / "nltk_data"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from course_data import dataset_file
import nltk
nltk.data.path.insert(0, os.environ["NLTK_DATA"])
random.seed(42)
np.random.seed(42)
print("Lab 5 | Abdulaziz Saqer BinRuqush | 2230007229")


Lab 5 | Abdulaziz Saqer BinRuqush | 2230007229


# Text Representation

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd


In [3]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [4]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [5]:
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [8]:
tfidf_vectorizer = TfidfVectorizer() # Create a CountVectorizer instance
vector_matrix = tfidf_vectorizer.fit_transform(data) # Fit and transform the documents into numerical vectors

In [9]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


In [10]:
import pandas as pd
import nltk
import numpy as np
import gensim

from nltk.tokenize import word_tokenize

In [11]:
df = pd.read_csv(dataset_file("news", "True.csv"))
print("True-news dataset:", df.shape)
display(df.head())


True-news dataset: (21417, 4)


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [12]:
# Re-iterable corpus: tokenize every document without retaining all tokens in RAM.
class NewsCorpus:
    def __iter__(self):
        for text in df["text"].dropna():
            yield word_tokenize(text)
tokens = NewsCorpus()


In [13]:
w2v = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100,
                            window=5, sg=1, seed=42, workers=1)
print("News vocabulary size:", len(w2v.wv))


News vocabulary size: 100446


In [14]:
print("Cosine similarity between 'provide' and 'program' - Skip Gram : ",w2v.wv.similarity('provide', 'program'))

Cosine similarity between 'provide' and 'program' - Skip Gram :  0.4253926


In [15]:
print("words that similar to 'program' - Skip Gram : ",w2v.wv.most_similar('program'))

words that similar to 'program' - Skip Gram :  [('programme', 0.8228200078010559), ('programs', 0.7991959452629089), ('modernization', 0.70235675573349), ('programmes', 0.6985149383544922), ('arsenal', 0.6869279742240906), ('curbed', 0.6867702007293701), ('accelerates', 0.6752763986587524), ('bond-buying', 0.6748735904693604), ('abolishing', 0.6720056533813477), ('curtail', 0.671937108039856)]


# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


In [16]:
sentences = ["This is the first document.", "This document is the second document.",
             "And this is the third one.", "Is this the first document?"]
sentence_tfidf = TfidfVectorizer()
sentence_vectors = sentence_tfidf.fit_transform(sentences)
similarities = cosine_similarity(sentence_vectors)
display(pd.DataFrame(similarities, index=sentences, columns=sentences))
assert np.allclose(similarities, similarities.T)
assert np.allclose(np.diag(similarities), 1.0)
print("The first and fourth sentences contain the same words, giving similarity:", similarities[0, 3])


,This is the first document.,This document is the second document.,And this is the third one.,Is this the first document?
This is the first document.,1.000000,0.646926,0.307772,1.000000
This document is the second document.,0.646926,1.000000,0.225240,0.646926
And this is the third one.,0.307772,0.225240,1.000000,0.307772
Is this the first document?,1.000000,0.646926,0.307772,1.000000


The first and fourth sentences contain the same words, giving similarity: 1.0


### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


In [17]:
task_sentences = ["data science is one of the most important fields of science",
                  "this is one of the best data science courses",
                  "data scientists analyze data"]
task_tfidf = TfidfVectorizer()
task_vectors = task_tfidf.fit_transform(task_sentences)
terms = task_tfidf.get_feature_names_out()
weights = pd.DataFrame(task_vectors.toarray(), columns=terms,
                       index=["Sentence 1", "Sentence 2", "Sentence 3"])
display(weights)
for sentence, row in weights.iterrows():
    print(sentence, "— largest TF-IDF weights:")
    print(row[row > 0].sort_values(ascending=False).head(5).to_string())


,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Sentence 1,0.000000,0.000000,0.000000,0.189526,0.320895,0.320895,0.244049,0.320895,0.488098,0.244049,0.488098,0.000000,0.244049,0.000000
Sentence 2,0.000000,0.400294,0.400294,0.236420,0.000000,0.000000,0.304434,0.000000,0.304434,0.304434,0.304434,0.000000,0.304434,0.400294
Sentence 3,0.542701,0.000000,0.000000,0.641055,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.542701,0.000000,0.000000


Sentence 1 — largest TF-IDF weights:
of           0.488098
science      0.488098
fields       0.320895
important    0.320895
most         0.320895
Sentence 2 — largest TF-IDF weights:
best       0.400294
courses    0.400294
this       0.400294
is         0.304434
of         0.304434
Sentence 3 — largest TF-IDF weights:
data          0.641055
analyze       0.542701
scientists    0.542701


## Word2vec

### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)

In [18]:
import re
simpsons = pd.read_csv(LAB_DIR / "dataset" / "simpsons_script_lines.csv",
                      usecols=["spoken_words"], low_memory=False)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,”“.’$-]", "", text)
    return text

simpsons_text = simpsons["spoken_words"].dropna().map(clean_text)
simpsons_text = simpsons_text[simpsons_text.str.strip().str.len() > 0]
class SimpsonsCorpus:
    def __iter__(self):
        for text in simpsons_text:
            yield word_tokenize(text)
Skip_gram_model = gensim.models.Word2Vec(
    SimpsonsCorpus(), min_count=1, vector_size=100, window=5, sg=1,
    seed=42, workers=1)
print("Rows in source:", len(simpsons))
print("Nonempty spoken lines:", len(simpsons_text))
print("Embedding dimensions:", Skip_gram_model.wv.vector_size)
print("Vocabulary size:", len(Skip_gram_model.wv))
assert Skip_gram_model.wv.vector_size == 100


Rows in source: 158314
Nonempty spoken lines: 131839
Embedding dimensions: 100
Vocabulary size: 44282


### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



In [19]:
for character in ["homer", "marge", "bart"]:
    print(f"Words most similar to {character}:")
    display(pd.DataFrame(Skip_gram_model.wv.most_similar(character), columns=["word", "similarity"]))


Words most similar to homer:


,word,similarity
0,abe,0.890125
1,marge,0.863329
2,bart,0.832169
3,grampa,0.831629
4,apu,0.819578
5,allison,0.819278
6,waylon,0.815572
7,ned,0.815088
8,barney,0.814847
9,jessica,0.814540


Words most similar to marge:


,word,similarity
0,homer,0.863329
1,abe,0.848489
2,sweetie,0.832057
3,becky,0.825736
4,sweetheart,0.818390
5,allison,0.815646
6,lurleen,0.812727
7,apu,0.808195
8,midge,0.807084
9,jessica,0.803007


Words most similar to bart:


,word,similarity
0,milhouse,0.883717
1,lisa,0.859538
2,grampa,0.854442
3,abe,0.845443
4,eliza,0.839740
5,sweetie,0.838817
6,jessica,0.838211
7,homer,0.832169
8,allison,0.830977
9,laddie,0.827767


### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

In [20]:
groups = [["jimbo", "milhouse", "kearney"],
          ["nelson", "bart", "milhouse"],
          ["homer", "patty", "selma"]]
for group in groups:
    print(group, "→ odd word:", Skip_gram_model.wv.doesnt_match(group))
print("Odd-word results reflect the learned usage contexts in this corpus.")


['jimbo', 'milhouse', 'kearney'] → odd word: milhouse
['nelson', 'bart', 'milhouse'] → odd word: nelson
['homer', 'patty', 'selma'] → odd word: homer
Odd-word results reflect the learned usage contexts in this corpus.
